In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel('INFO')
tf.random.set_seed(1)

In [ ]:
def separate_files(path, out_dir='/kaggle/working/train'):
    df = pd.read_csv(path)
    y = df['label']
    X = df.drop(columns='label')

    os.makedirs(out_dir, exist_ok=True)
    X.to_csv(f'{out_dir}_X.csv', index=False)
    y.to_csv(f'{out_dir}_y.csv', index=False)
    

In [ ]:
separate_files('/kaggle/input/digit-recognizer/train.csv')

In [ ]:
def load_data(x_path, y_path, batch_size=64, dataset_size=28000, train=True):
    AUTOTUNE = tf.data.AUTOTUNE
    train_size = int(0.8 * dataset_size)
    
    ds_x = tf.data.TextLineDataset(x_path).skip(1)
    ds_y = tf.data.TextLineDataset(y_path).skip(1)\
            .map(lambda y: tf.strings.to_number(y, tf.int32), num_parallel_calls=AUTOTUNE)\
            .map(lambda y: tf.one_hot(y, 10))\
            
    def sparse_x(line):
        fields = tf.strings.split(line, ',')
        fields = tf.strings.to_number(fields, tf.float32)
        fields = fields / 255.0
        fields = tf.reshape(fields, [28,28])
        return fields
    
    ds_x = ds_x.map(sparse_x, num_parallel_calls=AUTOTUNE)
    
    ds = tf.data.Dataset.zip((ds_x, ds_y))
    ds = ds.shuffle(dataset_size).batch(batch_size)
    
    if(train):
        ds_train = ds.take(train_size // batch_size).prefetch(1)
        ds_val = ds.skip(train_size // batch_size).prefetch(1)
        return ds_train, ds_val
    else:
        return ds_x.batch(batch_size).prefetch(1)
    

In [ ]:
x_path = '/kaggle/working/train_X.csv'
y_path = '/kaggle/working/train_y.csv'

ds_train, ds_val = load_data(x_path, y_path)

In [ ]:
type(ds_train)

In [ ]:
for x,y in ds_train.take(1):
    print(x[0])
    print(y[0])
    print()

In [ ]:
initializer = keras.initializers.HeNormal()
regularizer = keras.regularizers.l2()

model = keras.Sequential([
    keras.layers.Input(shape=(28,28,1))
])
for i in range(1,4):
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Conv2D(filters=36*(2**i),
                                 kernel_size=3 + (i - 1),
                                 activation='tanh',
                                 kernel_initializer=initializer,
                                 padding='same'))
    model.add(keras.layers.MaxPooling2D())

model.add(keras.layers.Flatten())
for i in range(3):
    model.add(keras.layers.Dense(150, activation='relu', kernel_initializer=initializer, kernel_regularizer=regularizer))

model.add(keras.layers.Dense(10,activation='softmax'))

In [ ]:
optimizer = keras.optimizers.Adam()

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy','f1_score']
)

model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(patience=5, min_delta=0.001, start_from_epoch=10, restore_best_weights=True)

history = model.fit(
    ds_train,
    validation_data=ds_val,
    callbacks=[early_stop],
    epochs=30
)

In [ ]:
df = pd.DataFrame(history.history)
df[['accuracy','val_accuracy']].plot()

In [ ]:
model.evaluate(ds_val)

In [ ]:
x_test_path = '/kaggle/input/digit-recognizer/test.csv'

ds_test = load_data(x_test_path, y_path, train=False)

In [ ]:
preds = model.predict(ds_test)

In [ ]:
pred_labels = np.argmax(preds, axis=1)
sub = pd.DataFrame({
    "ImageId": np.arange(1, len(pred_labels) + 1),
    "Label": pred_labels
})
sub.to_csv("submission.csv", index=False)